# EXP21D-v5 — Active Weighted Sampler Correctness
Single-seed correctness rerun for Uniform, Topology, and Importance samplers at fan-out `[15, 10]`. This notebook is prepared locally and is not pushed automatically.


In [ ]:
%%writefile requirements.txt
numpy>=1.26
pandas>=2.1
scikit-learn>=1.4
matplotlib>=3.8
PyYAML>=6.0
torch>=2.2
torch-geometric>=2.5
pyg_lib>=0.4
xgboost>=2.0
tabulate>=0.9


In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP21D"
  version: "v5_sampler_correctness"
  seed: 42
  evaluation_seed: 1042
data:
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016
time_window:
  reference_date: "2018-01-01"
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60
features:
  mode: "clean"
  use_cyclical_time: true
  use_amount_log: true
  exclude_high_cardinality: true
sampling:
  method: "uniform"
  undersampling_ratio: 1.0
  weight_attr: "sampling_weight"
  smoothing_alpha: 0.1
model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "mean"
  dropout: 0.2
training:
  batch_size: 1024
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  label_smoothing: 0.0
  grad_clip_norm: 5.0
  warmup_epochs: 0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 7
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
  log_step_interval: 1
  num_neighbors: [15, 10]
classifier:
  type: "xgboost"
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"
  tree_method: "hist"
threshold_calibration:
  selection_metric: "val_f1"
  min_precision_constraint: 0.15
ablation:
  run_feature_subsets: true
reproducibility:
  deterministic: true
  warn_only: true
output:
  model_dir: "/kaggle/working/model/exp21d_v5_sampler_correctness"
  result_dir: "/kaggle/working/result/exp21d_v5_sampler_correctness"
  save_feature_matrices: true


In [ ]:
%%writefile config.py
from pathlib import Path

import yaml


def load_config(config_path: str = "config.reference.yaml") -> dict:
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Berkas konfigurasi tidak ditemukan: {config_path}")
    with path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    if config.get("training", {}).get("num_neighbors") != [15, 10]:
        raise ValueError("EXP21D-v5 mengunci num_neighbors=[15, 10]")
    if config.get("sampling", {}).get("weight_attr") != "sampling_weight":
        raise ValueError("EXP21D-v5 mewajibkan weight_attr='sampling_weight'")
    return config


In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler

def load_and_split_raw_data(
    file_path: str, 
    start_year: int = 2016, 
    nrows: int = None,
    reference_date: str = '2018-01-01',
    train_days: int = 320,
    val_days: int = 40,
    gap_days: int = 8,
    test_days: int = 40
) -> tuple:
    """
    Memuat dataset mentah IBM Credit Card Transactions, memfilter era modern (Year >= start_year),
    melakukan parsing datetime kronologis, dan memotongnya ke dalam 4 partisi temporal SHINE FRAUD II
    SEBELUM proses rekayasa fitur atau penskalaan (Strict Out-of-Sample Protocol).
    """
    print(f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})...")
    
    # 1. Pemuatan chunked efisien jika start_year ditentukan
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000
        
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            filtered_chunk = chunk[chunk['Year'] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break
                    
        if not chunks:
            raise ValueError(f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}")
            
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data mentah...")
    print("Parsing timestamps...")
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("Sorting chronologically...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("Cleaning Amount & Target Label...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    # 2. Pembagian Jendela Waktu Temporal (SHINE FRAUD II)
    print("\nMelakukan pembagian temporal window strictly chronologic...")
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)
    
    train_df = df[(df['datetime'] >= t0) & (df['datetime'] < t_val_start)].copy().reset_index(drop=True)
    val_df = df[(df['datetime'] >= t_val_start) & (df['datetime'] < t_gap_start)].copy().reset_index(drop=True)
    gap_df = df[(df['datetime'] >= t_gap_start) & (df['datetime'] < t_test_start)].copy().reset_index(drop=True)
    test_df = df[(df['datetime'] >= t_test_start) & (df['datetime'] < t_test_end)].copy().reset_index(drop=True)
    
    def compute_split_stats(name: str, split_df: pd.DataFrame, start_dt, end_dt):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0,
                "fraud_rate_pct": 0.0,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0
            }
        n_fraud = int(split_df['Is Fraud?'].sum())
        n_users = int(split_df['User'].nunique())
        n_merchants = int(split_df['Merchant Name'].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4),
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)
    
    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment (EXP21D Golden Master)",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats("Validation", val_df, t_val_start, t_gap_start),
            "embargo_gap": compute_split_stats("Embargo Gap", gap_df, t_gap_start, t_test_start),
            "test": compute_split_stats("Test", test_df, t_test_start, t_test_end),
            "total_window": compute_split_stats("Total Window", total_window_df, t0, t_test_end)
        }
    }
    
    return train_df, val_df, test_df, gap_df, stats_dict

def apply_strict_anti_leakage_preprocessing(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_df: pd.DataFrame = None,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True,
    use_amount_log: bool = True
) -> tuple:
    """
    Menerapkan Strict Anti-Leakage Preprocessing Protocol (EXP21D Golden Master):
    1. StandardScaler Amount HANYA di-fit pada train_df, lalu mentransformasikan val_df, test_df, gap_df.
    2. Encoding kategorikal HANYA mempelajari kamus nilai unik dari train_df. Nilai baru pada val/test dipetakan ke 0 (<oov>).
    3. Mengeliminasi Zip dan Merchant City (mode: clean) untuk mencegah pohon keputusan menghafal ID kuasi.
    4. Menambahkan fitur waktu siklik (sin/cos jam & hari) yang bebas memorization.
    5. Menambahkan amount_log (log1p Amount) untuk menstabilkan distribusi numerik.
    """
    print(f"\n[STRICT ANTI-LEAKAGE] Menerapkan preprocessing (Mode: {feature_mode}, Cyclical Time: {use_cyclical_time}, Amount Log: {use_amount_log})...")
    
    # Simpan nilai Amount mentah positif untuk log amount
    for df_split in [train_df, val_df, test_df] + ([gap_df] if gap_df is not None and len(gap_df) > 0 else []):
        if use_amount_log:
            df_split['amount_log'] = np.log1p(np.maximum(0.0, df_split['Amount'].values)).astype(np.float32)

    # Tentukan atribut kategorikal yang diizinkan
    if feature_mode == "clean":
        cat_columns = ['Use Chip', 'Merchant State', 'MCC', 'Errors?']
        print("  • Fitur High-Cardinality ('Zip', 'Merchant City') DIHAPUS dari input!")
    elif feature_mode == "clean_basic":
        cat_columns = ['Use Chip', 'MCC', 'Errors?']
        print(f"  • Menggunakan fitur kategorikal minimal: {cat_columns}")
    else:
        raise ValueError(f"Feature mode '{feature_mode}' tidak dikenal. Pilih 'clean' atau 'clean_basic'.")

    # A. Standarisasi Fitur Numerik (Amount) — FIT HANYA DI TRAIN
    scaler_amount = StandardScaler()
    train_df['Amount'] = scaler_amount.fit_transform(train_df[['Amount']])
    val_df['Amount'] = scaler_amount.transform(val_df[['Amount']])
    test_df['Amount'] = scaler_amount.transform(test_df[['Amount']])
    if gap_df is not None and len(gap_df) > 0:
        gap_df['Amount'] = scaler_amount.transform(gap_df[['Amount']])
        
    print(f"  • Amount Scaler fit pada Train: mean={scaler_amount.mean_[0]:.4f}, std={scaler_amount.scale_[0]:.4f}")

    if use_amount_log:
        scaler_log = StandardScaler()
        train_df['amount_log'] = scaler_log.fit_transform(train_df[['amount_log']])
        val_df['amount_log'] = scaler_log.transform(val_df[['amount_log']])
        test_df['amount_log'] = scaler_log.transform(test_df[['amount_log']])
        if gap_df is not None and len(gap_df) > 0:
            gap_df['amount_log'] = scaler_log.transform(gap_df[['amount_log']])
        print(f"  • Amount Log Scaler fit pada Train: mean={scaler_log.mean_[0]:.4f}, std={scaler_log.scale_[0]:.4f}")

    # B. Robust Categorical Encoding — DICTIONARY FIT HANYA DI TRAIN
    categorical_encoders = {}
    for col in cat_columns:
        train_df[col] = train_df[col].fillna('<missing>').astype(str)
        val_df[col] = val_df[col].fillna('<missing>').astype(str)
        test_df[col] = test_df[col].fillna('<missing>').astype(str)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].fillna('<missing>').astype(str)
            
        # Pelajari mapping unik dari TRAIN saja
        unique_train_vals = sorted(train_df[col].unique().tolist())
        # Index 0 dicadangkan untuk OOV (<unknown>), kategori train mulai dari index 1
        cat_to_id = {val: idx + 1 for idx, val in enumerate(unique_train_vals)}
        categorical_encoders[col] = cat_to_id
        
        # Transform dengan fallback OOV = 0
        train_df[col] = train_df[col].map(cat_to_id).fillna(0).astype(int)
        val_df[col] = val_df[col].map(cat_to_id).fillna(0).astype(int)
        test_df[col] = test_df[col].map(cat_to_id).fillna(0).astype(int)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].map(cat_to_id).fillna(0).astype(int)
            
        num_oov_val = int((val_df[col] == 0).sum())
        num_oov_test = int((test_df[col] == 0).sum())
        print(f"  • Kolom '{col}': {len(cat_to_id)} kategori train dipelajari (OOV di Val: {num_oov_val}, di Test: {num_oov_test})")

    # C. Fitur Temporal Siklik (Cyclical Time Features)
    feature_cols = ['Amount']
    if use_amount_log:
        feature_cols.append('amount_log')
    feature_cols = feature_cols + cat_columns

    if use_cyclical_time:
        all_splits = [train_df, val_df, test_df]
        if gap_df is not None and len(gap_df) > 0:
            all_splits.append(gap_df)
            
        for split_df in all_splits:
            hour_float = split_df['datetime'].dt.hour + split_df['datetime'].dt.minute / 60.0
            split_df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24.0).astype(np.float32)
            split_df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24.0).astype(np.float32)
            
            dow = split_df['datetime'].dt.dayofweek
            split_df['dow_sin'] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            split_df['dow_cos'] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)

        time_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
        feature_cols = feature_cols + time_cols
        print(f"  • Ditambahkan 4 fitur waktu siklik: {time_cols}")

    print(f"Total fitur transaksi bersih (feature_cols): {len(feature_cols)} fitur -> {feature_cols}")
    
    artifacts = {
        "scaler_amount": scaler_amount,
        "categorical_encoders": categorical_encoders,
        "feature_cols": feature_cols,
        "feature_mode": feature_mode
    }
    
    return train_df, val_df, test_df, gap_df, feature_cols, artifacts



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21d & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list, scalers_dict: dict = None) -> tuple:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi bersih (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi
      - merchant: Agregat rata-rata Amount & jumlah transaksi
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)

    Anti-Leakage Guard:
    Jika scalers_dict is None, scaler untuk User dan Merchant di-fit pada data saat ini (hanya untuk Train).
    Jika scalers_dict disediakan, scaler dari Train digunakan untuk me-transform User dan Merchant
    pada Validation dan Test tanpa membocorkan statistik masa depan.
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id').reset_index(drop=True)
    
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id').reset_index(drop=True)
    
    # Penskalaan fitur User & Merchant (Strict Anti-Leakage)
    if scalers_dict is None:
        scaler_user = StandardScaler()
        user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
        
        scaler_merchant = StandardScaler()
        merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        
        out_scalers = {
            'scaler_user': scaler_user,
            'scaler_merchant': scaler_merchant
        }
    else:
        scaler_user = scalers_dict['scaler_user']
        scaler_merchant = scalers_dict['scaler_merchant']
        user_feats_scaled = scaler_user.transform(user_agg[['user_avg_amount', 'user_tx_count']])
        merchant_feats_scaled = scaler_merchant.transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        out_scalers = scalers_dict
        
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    # 3. Fitur Transaksi & Label
    tx_features = torch.tensor(df[feature_cols].values, dtype=torch.float)
    tx_labels = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    
    data['transaction'].x = tx_features
    data['transaction'].y = tx_labels
    data['transaction'].num_nodes = num_transactions
    
    data['user'].x = user_features
    data['user'].num_nodes = num_users
    
    data['merchant'].x = merchant_features
    data['merchant'].num_nodes = num_merchants
    
    # 4. Membangun Edge Index Bipartit
    tx_indices = np.arange(num_transactions)
    user_src_indices = df['User'].map(user_mapping).values
    merchant_src_indices = df['Merchant Name'].map(merchant_mapping).values
    
    edge_user_to_tx = torch.tensor(np.array([user_src_indices, tx_indices]), dtype=torch.long)
    edge_tx_to_user = torch.tensor(np.array([tx_indices, user_src_indices]), dtype=torch.long)
    
    edge_merchant_to_tx = torch.tensor(np.array([merchant_src_indices, tx_indices]), dtype=torch.long)
    edge_tx_to_merchant = torch.tensor(np.array([tx_indices, merchant_src_indices]), dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_user_to_tx
    data['transaction', 'rev_buys', 'user'].edge_index = edge_tx_to_user
    
    data['merchant', 'sells', 'transaction'].edge_index = edge_merchant_to_tx
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_tx_to_merchant
    
    print(f"HeteroData selesai: {num_transactions} tx, {num_users} users, {num_merchants} merchants.")
    return data, out_scalers



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self, 
        hidden_channels: int, 
        out_channels: int, 
        num_layers: int, 
        data_metadata: tuple, 
        feature_dims: dict, 
        aggr: str = 'mean',
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear bawaan untuk monitoring val AUPRC selama pelatihan GNN
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal
        h_dict = {}
        
        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
            elif node_type in self.param_embeddings:
                num_nodes = num_nodes_dict[node_type]
                h_dict[node_type] = self.param_embeddings[node_type].expand(num_nodes, -1)
                
        # 2. Message Passing Layer
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            if 'transaction' in h_dict and h_dict['transaction'].size(0) > 1:
                h_dict['transaction'] = self.bns[i](h_dict['transaction'])
            h_dict = {k: F.relu(v) for k, v in h_dict.items()}
            h_dict = {k: self.dropout(v) for k, v in h_dict.items()}
            
        tx_emb = h_dict['transaction']
        
        if return_embeddings:
            return tx_emb
            
        out = self.classifier(tx_emb)
        return out.squeeze(-1)

def manual_weighted_bce(logits, targets, pos_weight):
    """
    Weighted Binary Cross Entropy Loss stabil dengan koefisien pos_weight teredam.
    """
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (1 + (pos_weight - 1) * targets) * (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()

def manual_unweighted_bce(logits, targets):
    """
    Standar Binary Cross Entropy Loss tanpa pembobotan untuk evaluasi objektif.
    """
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()



In [ ]:
%%writefile samplers.py
from __future__ import annotations

from typing import Any

import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader


WEIGHT_ATTR = "sampling_weight"
USER_TO_TX = ("user", "buys", "transaction")
TX_TO_USER = ("transaction", "rev_buys", "user")
MERCHANT_TO_TX = ("merchant", "sells", "transaction")
TX_TO_MERCHANT = ("transaction", "rev_sells", "merchant")
WEIGHTED_RELATIONS = (TX_TO_USER, TX_TO_MERCHANT)


def counterpart_source(edge_type: tuple[str, str, str]) -> str:
    if edge_type == TX_TO_USER:
        return "merchant_degree"
    if edge_type == TX_TO_MERCHANT:
        return "user_degree"
    return "not_applicable"


def _bincount(index: torch.Tensor, size: int) -> torch.Tensor:
    return torch.bincount(index, minlength=size).to(dtype=torch.float32)


def _minmax_plus_floor(values: torch.Tensor, alpha: float) -> torch.Tensor:
    """Implement the predeclared min-max score + alpha formula."""
    if not 0.0 < alpha <= 1.0:
        raise ValueError(f"alpha harus berada pada (0, 1], diterima {alpha}")
    values = values.to(dtype=torch.float32)
    if values.numel() == 0:
        return values
    minimum, maximum = values.min(), values.max()
    if torch.isclose(maximum, minimum):
        return torch.ones_like(values)
    return (values - minimum) / (maximum - minimum + 1e-8) + alpha


def candidate_score(candidate_degree: torch.Tensor, method: str, alpha: float = 0.1) -> torch.Tensor:
    """Score competing transaction candidates from their counterpart degree."""
    method = method.lower()
    degree = candidate_degree.to(dtype=torch.float32).clamp_min(1.0)
    if method == "uniform":
        return torch.ones_like(degree)
    if method == "topology":
        raw = 1.0 / torch.log1p(degree)
    elif method == "importance":
        raw = torch.log1p(degree)
    else:
        raise ValueError(f"Metode sampling tidak didukung: {method}")
    return _minmax_plus_floor(raw, alpha)


def _transaction_counterpart_degrees(data: HeteroData) -> tuple[torch.Tensor, torch.Tensor]:
    required = {USER_TO_TX, TX_TO_USER, MERCHANT_TO_TX, TX_TO_MERCHANT}
    missing = required.difference(data.edge_types)
    if missing:
        raise ValueError(f"Relasi graf tripartit belum lengkap: {sorted(missing)}")

    transaction_store = data["transaction"]
    expected = int(transaction_store.num_nodes)
    if hasattr(transaction_store, "historical_user_degree") and hasattr(transaction_store, "historical_merchant_degree"):
        user_degree = transaction_store.historical_user_degree.to(torch.float32)
        merchant_degree = transaction_store.historical_merchant_degree.to(torch.float32)
        if user_degree.numel() != expected or merchant_degree.numel() != expected:
            raise ValueError("Shape historical counterpart degree tidak sama dengan jumlah transaksi")
        return user_degree, merchant_degree

    user_edges = data[USER_TO_TX].edge_index
    merchant_edges = data[MERCHANT_TO_TX].edge_index
    user_incoming = torch.bincount(user_edges[1], minlength=expected)
    merchant_incoming = torch.bincount(merchant_edges[1], minlength=expected)
    if not torch.equal(user_incoming, torch.ones_like(user_incoming)):
        raise ValueError("Setiap transaksi harus terhubung tepat ke satu user")
    if not torch.equal(merchant_incoming, torch.ones_like(merchant_incoming)):
        raise ValueError("Setiap transaksi harus terhubung tepat ke satu merchant")

    user_degree = _bincount(user_edges[0], int(data["user"].num_nodes))
    merchant_degree = _bincount(merchant_edges[0], int(data["merchant"].num_nodes))
    user_for_transaction = torch.empty(expected, dtype=torch.long)
    merchant_for_transaction = torch.empty(expected, dtype=torch.long)
    user_for_transaction[user_edges[1]] = user_edges[0]
    merchant_for_transaction[merchant_edges[1]] = merchant_edges[0]
    return user_degree[user_for_transaction], merchant_degree[merchant_for_transaction]


def _correlation(x: torch.Tensor, y: torch.Tensor) -> float | None:
    x, y = x.to(torch.float32), y.to(torch.float32)
    if x.numel() < 2 or torch.isclose(x.std(unbiased=False), torch.tensor(0.0)) or torch.isclose(y.std(unbiased=False), torch.tensor(0.0)):
        return None
    return float(torch.corrcoef(torch.stack([x, y]))[0, 1])


def apply_sampling_weights(
    data: HeteroData,
    method: str,
    alpha: float = 0.1,
    weight_attr: str = WEIGHT_ATTR,
) -> HeteroData:
    """
    Attach active edge weights for the second-hop transaction competition.

    Incoming edges sampled at a user destination use each candidate transaction's
    merchant degree. Incoming edges sampled at a merchant destination use each
    candidate transaction's user degree.
    """
    method = method.lower()
    user_degree_per_tx, merchant_degree_per_tx = _transaction_counterpart_degrees(data)
    for edge_type in data.edge_types:
        edge_index = data[edge_type].edge_index
        num_edges = int(edge_index.size(1))
        candidate_degree = torch.ones(num_edges, dtype=torch.float32)
        if edge_type == TX_TO_USER:
            candidate_degree = merchant_degree_per_tx[edge_index[0]]
        elif edge_type == TX_TO_MERCHANT:
            candidate_degree = user_degree_per_tx[edge_index[0]]
        weights = candidate_score(candidate_degree, method, alpha) if edge_type in WEIGHTED_RELATIONS else torch.ones(num_edges, dtype=torch.float32)
        if weights.numel() != num_edges or not torch.isfinite(weights).all() or (weights <= 0).any():
            raise ValueError(f"Bobot sampling invalid untuk relasi {edge_type}")
        data[edge_type][weight_attr] = weights.contiguous()
        data[edge_type].candidate_degree = candidate_degree.contiguous()
    validate_sampling_weight_contract(data, method, weight_attr)
    return data


def validate_sampling_weight_contract(data: HeteroData, method: str, weight_attr: str = WEIGHT_ATTR) -> None:
    method = method.lower()
    for edge_type in data.edge_types:
        store = data[edge_type]
        if weight_attr not in store:
            raise ValueError(f"{weight_attr} tidak ditemukan pada {edge_type}")
        weights = store[weight_attr]
        destination = store.edge_index[1]
        sums = torch.zeros(int(data[edge_type[2]].num_nodes), dtype=torch.float32)
        sums.scatter_add_(0, destination, weights)
        active = torch.bincount(destination, minlength=sums.numel()) > 0
        if active.any() and not (sums[active] > 0).all():
            raise ValueError(f"Jumlah bobot lokal non-positif pada {edge_type}")
        if edge_type not in WEIGHTED_RELATIONS and not torch.allclose(weights, torch.ones_like(weights)):
            raise ValueError(f"Relasi first-hop/non-kompetitif harus uniform: {edge_type}")
        if edge_type in WEIGHTED_RELATIONS:
            degree = store.candidate_degree.clamp_min(1.0)
            varies = torch.unique(degree).numel() > 1
            if method == "uniform" and not torch.allclose(weights, torch.ones_like(weights)):
                raise ValueError(f"Uniform sampler harus berbobot satu pada {edge_type}")
            if method != "uniform" and varies and torch.unique(weights).numel() <= 1:
                raise ValueError(f"Bobot {method} konstan meski counterpart degree bervariasi pada {edge_type}")
            correlation = _correlation(degree, weights)
            if method == "topology" and correlation is not None and correlation >= 0:
                raise ValueError(f"Topology weight harus menurun terhadap counterpart degree pada {edge_type}")
            if method == "importance" and correlation is not None and correlation <= 0:
                raise ValueError(f"Importance weight harus meningkat terhadap counterpart degree pada {edge_type}")


def sampling_weight_audit(data: HeteroData, method: str, weight_attr: str = WEIGHT_ATTR) -> dict[str, Any]:
    report: dict[str, Any] = {
        "method": method,
        "weight_attr": weight_attr,
        "weighted_relations": ["__".join(item) for item in WEIGHTED_RELATIONS],
        "relations": {},
    }
    for edge_type in data.edge_types:
        store = data[edge_type]
        weights = store[weight_attr].detach().cpu()
        degrees = store.candidate_degree.detach().cpu()
        report["relations"]["__".join(edge_type)] = {
            "num_edges": int(weights.numel()),
            "counterpart_source": counterpart_source(edge_type),
            "weight_min": float(weights.min()) if weights.numel() else None,
            "weight_max": float(weights.max()) if weights.numel() else None,
            "weight_mean": float(weights.mean()) if weights.numel() else None,
            "weight_std": float(weights.std(unbiased=False)) if weights.numel() else None,
            "unique_weights": int(torch.unique(weights).numel()),
            "candidate_degree_min": float(degrees.min()) if degrees.numel() else None,
            "candidate_degree_max": float(degrees.max()) if degrees.numel() else None,
            "degree_weight_correlation": _correlation(degrees.clamp_min(1.0), weights),
            "contract_passed": True,
        }
    return report


def get_neighbor_loader(
    data: HeteroData,
    method: str = "uniform",
    batch_size: int = 1024,
    shuffle: bool = False,
    num_neighbors: list[int] | None = None,
    seed: int = 42,
    alpha: float = 0.1,
    input_nodes: torch.Tensor | None = None,
    weight_attr: str = WEIGHT_ATTR,
) -> NeighborLoader:
    if num_neighbors is None:
        num_neighbors = [15, 10]
    if len(num_neighbors) != 2 or any(value <= 0 for value in num_neighbors):
        raise ValueError("num_neighbors harus berisi dua fan-out positif")
    apply_sampling_weights(data, method, alpha, weight_attr)
    if input_nodes is None:
        input_nodes = torch.arange(int(data["transaction"].num_nodes))
    generator = torch.Generator().manual_seed(seed)
    loader = NeighborLoader(
        data,
        num_neighbors={edge_type: list(num_neighbors) for edge_type in data.edge_types},
        batch_size=batch_size,
        input_nodes=("transaction", input_nodes),
        shuffle=shuffle,
        num_workers=0,
        weight_attr=weight_attr,
        generator=generator,
        subgraph_type="directional",
    )
    return loader


In [ ]:
%%writefile validate_sampler.py
from __future__ import annotations

import json
from types import SimpleNamespace

import torch
from torch_geometric.data import HeteroData

import samplers as sampler_module
from samplers import (
    MERCHANT_TO_TX,
    TX_TO_MERCHANT,
    TX_TO_USER,
    USER_TO_TX,
    apply_sampling_weights,
    get_neighbor_loader,
    sampling_weight_audit,
)


def toy_graph() -> HeteroData:
    data = HeteroData()
    data["transaction"].x = torch.randn(6, 10)
    data["transaction"].y = torch.tensor([0, 1, 0, 0, 1, 0], dtype=torch.float32)
    data["user"].x = torch.randn(3, 2)
    data["merchant"].x = torch.randn(3, 2)
    transactions = torch.arange(6)
    users = torch.tensor([0, 0, 0, 1, 2, 2])
    merchants = torch.tensor([0, 1, 1, 1, 2, 2])
    data[USER_TO_TX].edge_index = torch.stack([users, transactions])
    data[TX_TO_USER].edge_index = torch.stack([transactions, users])
    data[MERCHANT_TO_TX].edge_index = torch.stack([merchants, transactions])
    data[TX_TO_MERCHANT].edge_index = torch.stack([transactions, merchants])
    return data


def main() -> None:
    uniform = toy_graph()
    topology = toy_graph()
    importance = toy_graph()
    apply_sampling_weights(uniform, "uniform")
    apply_sampling_weights(topology, "topology")
    apply_sampling_weights(importance, "importance")

    topology_user_weights = topology[TX_TO_USER].sampling_weight
    importance_user_weights = importance[TX_TO_USER].sampling_weight
    original_importance = {edge_type: importance[edge_type].sampling_weight.clone() for edge_type in importance.edge_types}
    importance["transaction"].y = 1 - importance["transaction"].y
    apply_sampling_weights(importance, "importance")
    topology_audit = sampling_weight_audit(topology, "topology")
    real_neighbor_loader = sampler_module.NeighborLoader

    class CaptureNeighborLoader:
        def __init__(self, *args, **kwargs):
            self.node_sampler = SimpleNamespace(weight_attr=kwargs.get("weight_attr"))
            self.kwargs = kwargs

    sampler_module.NeighborLoader = CaptureNeighborLoader
    try:
        loader = get_neighbor_loader(toy_graph(), "importance", batch_size=2, num_neighbors=[2, 2], seed=42)
    finally:
        sampler_module.NeighborLoader = real_neighbor_loader
    active_weight_attr = getattr(loader.node_sampler, "weight_attr", None)
    duplicate = toy_graph()
    duplicate[USER_TO_TX].edge_index = torch.cat([duplicate[USER_TO_TX].edge_index, duplicate[USER_TO_TX].edge_index[:, :1]], dim=1)
    duplicate_rejected = False
    try:
        apply_sampling_weights(duplicate, "topology")
    except ValueError:
        duplicate_rejected = True

    checks = {
        "uniform_weights_are_one": all(torch.allclose(uniform[edge].sampling_weight, torch.ones_like(uniform[edge].sampling_weight)) for edge in uniform.edge_types),
        "tx_to_user_uses_merchant_degree": topology_audit["relations"]["transaction__rev_buys__user"]["counterpart_source"] == "merchant_degree",
        "tx_to_merchant_uses_user_degree": topology_audit["relations"]["transaction__rev_sells__merchant"]["counterpart_source"] == "user_degree",
        "topology_decreases_with_degree": bool(topology_user_weights[0] > topology_user_weights[1]),
        "importance_increases_with_degree": bool(importance_user_weights[0] < importance_user_weights[1]),
        "first_hop_remains_uniform": torch.allclose(topology[USER_TO_TX].sampling_weight, torch.ones(6)) and torch.allclose(topology[MERCHANT_TO_TX].sampling_weight, torch.ones(6)),
        "labels_do_not_affect_weights": all(torch.equal(original_importance[edge], importance[edge].sampling_weight) for edge in importance.edge_types),
        "duplicate_counterpart_rejected": duplicate_rejected,
        "neighbor_loader_uses_weight_attr": active_weight_attr == "sampling_weight",
        "audit_reports_contract": all(item["contract_passed"] for item in topology_audit["relations"].values()),
    }
    report = {"status": "passed" if all(checks.values()) else "failed", "checks": checks}
    print(json.dumps(report, indent=2))
    if report["status"] != "passed":
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile main.py
import argparse
import time
import json
import os
import random
import platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
import torch.optim as optim
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    accuracy_score, roc_auc_score, average_precision_score, 
    confusion_matrix
)
from xgboost import XGBClassifier

from config import load_config
from data_preprocessing import load_and_split_raw_data, apply_strict_anti_leakage_preprocessing
from graph_builder import build_hetero_graph, graph_level_undersample
from samplers import (
    WEIGHT_ATTR,
    apply_sampling_weights,
    get_neighbor_loader,
    sampling_weight_audit,
)
from model import HeteroGraphSAGE, manual_weighted_bce, manual_unweighted_bce


def seed_everything(seed, deterministic=True, warn_only=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=warn_only)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
    temporary.replace(path)


def append_event(path, event, **payload):
    record = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "event": event, **payload}
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

def extract_features(model, loader, device, total_samples, orig_feat_dim, emb_dim):
    """
    Mengekstrak embedding GNN dan menggabungkannya dengan fitur tabular transaksi yang bersih.
    Menggunakan pre-alokasi memori numpy untuk efisiensi RAM maksimal.
    """
    model.eval()
    final_features = np.empty((total_samples, orig_feat_dim + emb_dim), dtype=np.float32)
    all_targets = np.empty((total_samples,), dtype=np.int64)
    
    start_time = time.time()
    current_idx = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            
            embeddings = model(
                batch.x_dict, 
                batch.edge_index_dict, 
                num_nodes_dict=num_nodes_dict, 
                return_embeddings=True
            )
            
            batch_size = batch['transaction'].batch_size
            next_idx = min(current_idx + batch_size, total_samples)
            actual_count = next_idx - current_idx
            
            final_features[current_idx:next_idx, :orig_feat_dim] = batch['transaction'].x[:actual_count].cpu().numpy()
            final_features[current_idx:next_idx, orig_feat_dim:] = embeddings[:actual_count].cpu().numpy()
            all_targets[current_idx:next_idx] = batch['transaction'].y[:actual_count].cpu().numpy()
            
            current_idx = next_idx
            
    inference_time = time.time() - start_time
    
    if current_idx < total_samples:
        final_features = final_features[:current_idx]
        all_targets = all_targets[:current_idx]
        
    return final_features, all_targets, inference_time

def find_optimal_threshold(val_probs, val_labels, thresholds=None, min_precision=0.15):
    """
    Mencari threshold optimal tau* yang memaksimalkan F1-score pada Validation set,
    dengan konstrain presisi minimum yang realistis untuk perbankan.
    """
    if thresholds is None:
        thresholds = np.arange(0.02, 0.75, 0.005)
        
    best_tau = 0.5
    best_f1 = 0.0
    best_metrics = {}
    
    for tau in thresholds:
        preds = (val_probs >= tau).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        prec = precision_score(val_labels, preds, zero_division=0)
        rec = recall_score(val_labels, preds, zero_division=0)
        
        if f1 > best_f1 and prec >= min_precision:
            best_f1 = f1
            best_tau = float(tau)
            best_metrics = {
                "tau": float(tau),
                "val_f1": float(f1),
                "val_precision": float(prec),
                "val_recall": float(rec)
            }
            
    # Fallback jika konstrain presisi terlalu ketat
    if best_f1 == 0.0:
        for tau in thresholds:
            preds = (val_probs >= tau).astype(int)
            f1 = f1_score(val_labels, preds, zero_division=0)
            prec = precision_score(val_labels, preds, zero_division=0)
            rec = recall_score(val_labels, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_tau = float(tau)
                best_metrics = {
                    "tau": float(tau),
                    "val_f1": float(f1),
                    "val_precision": float(prec),
                    "val_recall": float(rec)
                }

    if best_f1 == 0.0:
        best_tau = 0.5
        best_metrics = {
            "tau": 0.5,
            "val_f1": 0.0,
            "val_precision": 0.0,
            "val_recall": 0.0
        }
        
    return best_tau, best_metrics

def evaluate_predictions(probs, labels, optimal_tau=0.5):
    """
    Menghitung metrik klasifikasi komprehensif pada probabilitas kontinu dan biner.
    """
    opt_preds = (probs >= optimal_tau).astype(int)
    f1_opt = float(f1_score(labels, opt_preds, zero_division=0))
    rec_opt = float(recall_score(labels, opt_preds, zero_division=0))
    prec_opt = float(precision_score(labels, opt_preds, zero_division=0))
    acc_opt = float(accuracy_score(labels, opt_preds))
    cm_opt = confusion_matrix(labels, opt_preds)
    
    if cm_opt.shape == (2, 2):
        tn, fp, fn, tp = cm_opt.ravel()
    else:
        tn = int(cm_opt[0, 0]) if len(cm_opt) > 0 else 0
        fp, fn, tp = 0, 0, 0

    spec_opt = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    gmean_opt = float(np.sqrt(rec_opt * spec_opt))
    
    # Evaluasi pada default tau = 0.50
    def_preds = (probs >= 0.5).astype(int)
    f1_def = float(f1_score(labels, def_preds, zero_division=0))
    rec_def = float(recall_score(labels, def_preds, zero_division=0))
    prec_def = float(precision_score(labels, def_preds, zero_division=0))
    
    # Metrik independen threshold
    try:
        auprc = float(average_precision_score(labels, probs))
    except Exception:
        auprc = 0.0
        
    try:
        roc_auc = float(roc_auc_score(labels, probs))
    except Exception:
        roc_auc = 0.5
        
    # Finansial / Perbankan: Lift@1% dan TP@500
    n_samples = len(labels)
    k_1pct = max(1, int(np.ceil(0.01 * n_samples)))
    top_1pct_idx = np.argsort(probs)[-k_1pct:]
    top_1pct_tp = int(labels[top_1pct_idx].sum())
    total_pos = int(labels.sum())
    baseline_rate = (total_pos / n_samples) if n_samples > 0 else 0.0
    model_rate = (top_1pct_tp / k_1pct) if k_1pct > 0 else 0.0
    lift_1pct = float(model_rate / baseline_rate) if baseline_rate > 0 else 1.0
    
    top_500_idx = np.argsort(probs)[-500:]
    tp_at_500 = int(labels[top_500_idx].sum())
    prec_at_500 = float(tp_at_500 / 500.0)
    
    return {
        "metrics_optimal_threshold": {
            "threshold": float(optimal_tau),
            "f1_score": f1_opt,
            "f1_macro": float(f1_score(labels, opt_preds, average='macro', zero_division=0)),
            "recall": rec_opt,
            "precision": prec_opt,
            "accuracy": acc_opt,
            "gmean": gmean_opt,
            "confusion_matrix": {
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp)
            }
        },
        "metrics_default_threshold_05": {
            "threshold": 0.5,
            "f1_score": f1_def,
            "recall": rec_def,
            "precision": prec_def
        },
        "metrics_threshold_independent": {
            "auprc": auprc,
            "roc_auc": roc_auc,
            "lift_1_percent": lift_1pct,
            "tp_at_500": tp_at_500,
            "precision_at_500": prec_at_500
        }
    }

def main():
    parser = argparse.ArgumentParser(description="EXP21D-v5: Active Weighted Sampler Correctness")
    parser.add_argument("--config", type=str, default="config.reference.yaml", help="Path to config YAML")
    parser.add_argument("--sampling", type=str, default=None, help="Neighbor sampling: uniform, topology, importance")
    parser.add_argument("--nrows", type=int, default=None, help="Limit rows for quick smoke test")
    parser.add_argument("--device", type=str, default=None, help="Device: cuda or cpu")
    parser.add_argument("--seed", type=int, default=None, help="Random seed")
    args = parser.parse_args()
    
    print("\n" + "="*80)
    print("  EKSEKUSI EXP21D-v5: ACTIVE WEIGHTED SAMPLER CORRECTNESS")
    print("="*80 + "\n")
    
    cfg = load_config(args.config)
    experiment_cfg = cfg.get('experiment', {})
    seed = int(args.seed if args.seed is not None else experiment_cfg.get('seed', 42))
    evaluation_seed = int(experiment_cfg.get('evaluation_seed', seed + 1000))
    repro_cfg = cfg.get('reproducibility', {})
    seed_everything(seed, repro_cfg.get('deterministic', True), repro_cfg.get('warn_only', True))
    start_total_time = time.time()
    
    sampling_method = args.sampling or cfg.get('sampling', {}).get('method', 'uniform')
    if sampling_method not in {"uniform", "topology", "importance"}:
        raise ValueError(f"Sampling method tidak didukung: {sampling_method}")
    device_name = args.device or ('cuda' if torch.cuda.is_available() else 'cpu')
    device = torch.device(device_name)
    print(f"Menggunakan perangkat komputasi: {device} ({torch.cuda.get_device_name(0) if device_name == 'cuda' else 'CPU'})")
    print(f"Metode Neighbor Sampling terpilih: {sampling_method.upper()}")
    
    # Direktori Output
    result_root = Path(cfg.get('output', {}).get('result_dir', '../../result/exp21d_v5_sampler_correctness'))
    model_root = Path(cfg.get('output', {}).get('model_dir', '../../model/exp21d_v5_sampler_correctness'))
    result_dir = result_root / f"seed_{seed}"
    model_dir = model_root / f"seed_{seed}"
    result_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    plots_dir = result_dir / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)
    events_path = result_dir / "events.jsonl"
    append_event(events_path, "run_started", method=sampling_method, seed=seed, device=str(device))
    write_json(result_dir / f"exp21d_v5_{sampling_method}_manifest.json", {
        "experiment_id": "EXP21D-v5-SamplerCorrectness",
        "status": "running",
        "sampling_method": sampling_method,
        "seed": seed,
        "evaluation_seed": evaluation_seed,
        "weight_attr": WEIGHT_ATTR,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "environment": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "cuda": torch.version.cuda,
            "cuda_available": torch.cuda.is_available(),
        },
    })
    
    # 1. Pemuatan dan Pembagian Data Strictly Chronological (SHINE FRAUD II)
    t_data_start = time.time()
    data_cfg = cfg['data']
    tw_cfg = cfg['time_window']
    feat_cfg = cfg.get('features', {})
    
    train_df, val_df, test_df, gap_df, stats_dict = load_and_split_raw_data(
        file_path=data_cfg['transactions'],
        start_year=data_cfg.get('start_year', 2016),
        nrows=args.nrows or data_cfg.get('max_rows', None),
        reference_date=tw_cfg.get('reference_date', '2018-01-01'),
        train_days=tw_cfg.get('train_days', 320),
        val_days=tw_cfg.get('val_days', 40),
        gap_days=tw_cfg.get('gap_days', 8),
        test_days=tw_cfg.get('test_days', 40)
    )
    
    # Simpan statistik partisi window
    write_json(result_dir / "window_statistics.json", stats_dict)
    print(f"[INFO] Statistik jendela waktu tersimpan di: {result_dir / 'window_statistics.json'}")
    
    # 2. Strict Anti-Leakage Preprocessing & Clean Attributes
    train_df, val_df, test_df, gap_df, feature_cols, art_prep = apply_strict_anti_leakage_preprocessing(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        gap_df=gap_df,
        feature_mode=feat_cfg.get('mode', 'clean'),
        use_cyclical_time=feat_cfg.get('use_cyclical_time', True),
        use_amount_log=feat_cfg.get('use_amount_log', True)
    )
    data_prep_time = time.time() - t_data_start
    
    # 3. Graph-Level Undersampling
    samp_ratio = cfg.get('sampling', {}).get('undersampling_ratio', 1.0)
    train_df = graph_level_undersample(train_df, ratio=samp_ratio, seed=seed)
    
    # 4. Pembangunan Graf Heterogen PyG
    train_graph, scalers_dict = build_hetero_graph(train_df, feature_cols, scalers_dict=None)
    val_graph, _ = build_hetero_graph(val_df, feature_cols, scalers_dict=scalers_dict)
    test_graph, _ = build_hetero_graph(test_df, feature_cols, scalers_dict=scalers_dict)
    
    # 5. Loader Pembuat Mini-Batch Neighbor Sampling
    t_cfg = cfg.get('training', {})
    batch_size = t_cfg.get('batch_size', 1024)
    num_neighbors = t_cfg.get('num_neighbors', [15, 10])
    sampling_cfg = cfg.get('sampling', {})
    alpha = float(sampling_cfg.get('smoothing_alpha', 0.1))
    weight_attr = sampling_cfg.get('weight_attr', WEIGHT_ATTR)
    graph_audits = {}
    for split_name, graph in (("train", train_graph), ("validation", val_graph), ("test", test_graph)):
        apply_sampling_weights(graph, sampling_method, alpha=alpha, weight_attr=weight_attr)
        graph_audits[split_name] = sampling_weight_audit(graph, sampling_method, weight_attr=weight_attr)
    audit_path = result_dir / f"exp21d_v5_{sampling_method}_sampler_weight_audit.json"
    write_json(audit_path, {
        "status": "passed",
        "correctness_gate": "counterpart_degree_and_active_weight_attr",
        "num_neighbors": num_neighbors,
        "alpha": alpha,
        "weight_attr": weight_attr,
        "graphs": graph_audits,
    })
    append_event(events_path, "sampler_correctness_gate_passed", audit=str(audit_path))

    train_loader = get_neighbor_loader(train_graph, method=sampling_method, batch_size=batch_size, shuffle=True, num_neighbors=num_neighbors, seed=seed, alpha=alpha, weight_attr=weight_attr)
    val_loader = get_neighbor_loader(val_graph, method=sampling_method, batch_size=batch_size, shuffle=False, num_neighbors=num_neighbors, seed=evaluation_seed, alpha=alpha, weight_attr=weight_attr)
    test_loader = get_neighbor_loader(test_graph, method=sampling_method, batch_size=batch_size, shuffle=False, num_neighbors=num_neighbors, seed=evaluation_seed + 1, alpha=alpha, weight_attr=weight_attr)
    
    # 6. Inisialisasi Model HeteroGraphSAGE
    m_cfg = cfg.get('model', {})
    hidden_dim = m_cfg.get('hidden_channels', 128)
    num_layers = m_cfg.get('num_layers', 2)
    aggr = m_cfg.get('aggr', 'mean')
    dropout = m_cfg.get('dropout', 0.2)
    
    feature_dims = {
        'transaction': len(feature_cols),
        'user': 2,
        'merchant': 2
    }
    
    model = HeteroGraphSAGE(
        hidden_channels=hidden_dim,
        out_channels=1,
        num_layers=num_layers,
        data_metadata=train_graph.metadata(),
        feature_dims=feature_dims,
        aggr=aggr,
        dropout=dropout
    ).to(device)
    
    # Inisialisasi bobot via dry run mini-batch
    for dry_batch in train_loader:
        dry_batch = dry_batch.to(device)
        num_nodes_dict = {ntype: dry_batch[ntype].num_nodes for ntype in dry_batch.node_types}
        _ = model(dry_batch.x_dict, dry_batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
        break
        
    # Optimizer & Scheduler
    lr = t_cfg.get('learning_rate', 0.001)
    weight_decay = t_cfg.get('weight_decay', 0.0001)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    scheduler = None
    if t_cfg.get('use_lr_scheduler', True):
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='max', 
            factor=t_cfg.get('lr_scheduler_factor', 0.5), 
            patience=t_cfg.get('lr_scheduler_patience', 3)
        )
        
    # Pembobotan Kelas Positif (Loss Sqrt Pos-Weight)
    n_neg = int((train_df['Is Fraud?'] == 0).sum())
    n_pos = int((train_df['Is Fraud?'] == 1).sum())
    full_pos_weight = float(n_neg / n_pos) if n_pos > 0 else 1.0
    sqrt_pos_weight = float(np.sqrt(full_pos_weight))
    loss_strategy = t_cfg.get('loss_weight_strategy', 'sqrt')
    if loss_strategy == 'sqrt':
        pos_weight = min(sqrt_pos_weight, t_cfg.get('max_pos_weight', 50.0))
    elif loss_strategy == 'full':
        pos_weight = full_pos_weight
    else:
        pos_weight = 1.0
    print(f"Konfigurasi Loss: strategy={loss_strategy}, pos_weight={pos_weight:.2f} (Full ratio: {full_pos_weight:.2f})")
    
    # 7. Pelatihan GNN dengan Early Stopping pada Validation AUPRC
    print("\n" + "="*50)
    print("  MEMULAI PELATIHAN HETEROGRAPHSAGE")
    print("="*50)
    
    epochs = t_cfg.get('epochs', 20)
    early_stopping_patience = t_cfg.get('early_stopping_patience', 7)
    best_val_auprc = 0.0
    best_epoch = 0
    patience_counter = 0
    best_model_state = None
    
    training_history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_auprc": [],
        "val_f1": []
    }
    
    t_gnn_start = time.time()
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_tx = 0
        
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            out = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            
            batch_size_cur = batch['transaction'].batch_size
            logits = out[:batch_size_cur]
            targets = batch['transaction'].y[:batch_size_cur]
            
            loss = manual_weighted_bce(logits, targets, pos_weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), t_cfg.get('grad_clip_norm', 5.0))
            optimizer.step()
            
            total_loss += float(loss.item()) * batch_size_cur
            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += int((preds == targets).sum().item())
            total_tx += batch_size_cur
            
        train_loss = total_loss / total_tx if total_tx > 0 else 0.0
        train_acc = total_correct / total_tx if total_tx > 0 else 0.0
        
        # Validasi per-epoch
        model.eval()
        val_loss_total = 0.0
        val_tx_count = 0
        all_val_probs = []
        all_val_targets = []
        
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
                out = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
                
                batch_size_cur = batch['transaction'].batch_size
                logits = out[:batch_size_cur]
                targets = batch['transaction'].y[:batch_size_cur]
                
                loss_v = manual_unweighted_bce(logits, targets)
                val_loss_total += float(loss_v.item()) * batch_size_cur
                val_tx_count += batch_size_cur
                
                probs = torch.sigmoid(logits).cpu().numpy()
                all_val_probs.append(probs)
                all_val_targets.append(targets.cpu().numpy())
                
        val_loss = val_loss_total / val_tx_count if val_tx_count > 0 else 0.0
        val_probs_arr = np.concatenate(all_val_probs)
        val_targets_arr = np.concatenate(all_val_targets)
        
        try:
            val_auprc = float(average_precision_score(val_targets_arr, val_probs_arr))
        except Exception:
            val_auprc = 0.0
            
        val_preds = (val_probs_arr >= 0.5).astype(int)
        val_f1 = float(f1_score(val_targets_arr, val_preds, zero_division=0))
        
        if scheduler is not None:
            scheduler.step(val_auprc)
            
        training_history["train_loss"].append(train_loss)
        training_history["train_acc"].append(train_acc)
        training_history["val_loss"].append(val_loss)
        training_history["val_auprc"].append(val_auprc)
        training_history["val_f1"].append(val_f1)
        training_history.setdefault("learning_rate", []).append(float(optimizer.param_groups[0]["lr"]))
        write_json(result_dir / f"exp21d_v5_{sampling_method}_history.json", training_history)
        append_event(
            events_path,
            "epoch_completed",
            method=sampling_method,
            epoch=epoch,
            train_loss=train_loss,
            val_loss=val_loss,
            val_auprc=val_auprc,
            val_f1=val_f1,
        )
        
        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val AUPRC: {val_auprc:.4f} | Val F1: {val_f1:.4f}")
        
        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            best_epoch = epoch
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"[EARLY STOPPING] Tidak ada peningkatan AUPRC selama {early_stopping_patience} epoch berturut-turut. Berhenti di epoch {epoch}.")
                break
                
    gnn_train_time = time.time() - t_gnn_start
    print(f"\n[SELESAI GNN] Best Validation AUPRC: {best_val_auprc:.4f} tercapai pada Epoch {best_epoch} (Waktu: {gnn_train_time:.1f}s)")
    
    # Pulihkan checkpoint terbaik
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
        
    # Simpan bobot GNN terbaik
    gnn_model_path = model_dir / f"exp21d_v5_{sampling_method}_best_gnn.pt"
    torch.save(model.state_dict(), gnn_model_path)
    print(f"[CHECKPOINT] Model GNN terbaik disimpan ke: {gnn_model_path}")
    
    # 8. Ekstraksi Fitur Hibrida (Tabular Bersih + 128-d GNN Embedding)
    print("\n" + "="*50)
    print("  EKSTRAKSI FITUR HIBRIDA (TABULAR + GNN EMBEDDING)")
    print("="*50)
    
    feat_dim_tab = len(feature_cols)
    train_extract_loader = get_neighbor_loader(
        train_graph,
        method=sampling_method,
        batch_size=batch_size,
        shuffle=False,
        num_neighbors=num_neighbors,
        seed=evaluation_seed + 2,
        alpha=alpha,
        weight_attr=weight_attr,
    )
    X_train, y_train, inf_train_t = extract_features(model, train_extract_loader, device, len(train_df), feat_dim_tab, hidden_dim)
    X_val, y_val, inf_val_t = extract_features(model, val_loader, device, len(val_df), feat_dim_tab, hidden_dim)
    X_test, y_test, inf_test_t = extract_features(model, test_loader, device, len(test_df), feat_dim_tab, hidden_dim)
    
    print(f"Fitur Hibrida terbentuk: {X_train.shape[1]} dimensi ({feat_dim_tab} tabular + {hidden_dim} GNN embedding).")
    
    # 9. Pelatihan Downstream XGBoost Classifier Ter-regularisasi
    print("\n" + "="*50)
    print("  PELATIHAN DOWNSTREAM XGBOOST CLASSIFIER")
    print("="*50)
    
    c_cfg = cfg.get('classifier', {})
    xgb_spw_strategy = c_cfg.get('scale_pos_weight_strategy', 'sqrt')
    if xgb_spw_strategy == 'sqrt':
        xgb_spw = sqrt_pos_weight
    elif xgb_spw_strategy == 'full':
        xgb_spw = full_pos_weight
    else:
        xgb_spw = 1.0
        
    xgb_params = {
        'n_estimators': c_cfg.get('n_estimators', 100),
        'max_depth': c_cfg.get('max_depth', 4),
        'learning_rate': c_cfg.get('learning_rate', 0.05),
        'min_child_weight': c_cfg.get('min_child_weight', 10),
        'subsample': c_cfg.get('subsample', 0.8),
        'colsample_bytree': c_cfg.get('colsample_bytree', 0.8),
        'scale_pos_weight': xgb_spw,
        'random_state': seed,
        'n_jobs': -1,
        'eval_metric': 'logloss',
        'tree_method': c_cfg.get('tree_method', 'hist')
    }
    print(f"XGBoost Parameters: {xgb_params}")
    
    t_xgb_start = time.time()
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    xgb_histories = {"hybrid": xgb_model.evals_result()}
    xgb_train_time = time.time() - t_xgb_start
    print(f"[SELESAI XGBOOST] Pelatihan selesai dalam {xgb_train_time:.2f} detik.")
    
    # Simpan model XGBoost
    xgb_model_path = model_dir / f"exp21d_v5_{sampling_method}_xgb.json"
    xgb_model.save_model(str(xgb_model_path))
    print(f"[CHECKPOINT] Model XGBoost disimpan ke: {xgb_model_path}")
    
    # 10. Kalibrasi Threshold Dinamis pada Validation Set
    val_probs = xgb_model.predict_proba(X_val)[:, 1]
    min_prec = cfg.get('threshold_calibration', {}).get('min_precision_constraint', 0.15)
    optimal_tau, tau_metrics = find_optimal_threshold(val_probs, y_val, min_precision=min_prec)
    print(f"\n[THRESHOLD CALIBRATION] Threshold optimal tau* = {optimal_tau:.4f} (Val F1: {tau_metrics.get('val_f1', 0):.4f}, Val Precision: {tau_metrics.get('val_precision', 0):.4f}, Val Recall: {tau_metrics.get('val_recall', 0):.4f})")
    
    # 11. Evaluasi Menyeluruh pada Test Set Masa Depan (60 Hari)
    test_probs = xgb_model.predict_proba(X_test)[:, 1]
    eval_results = evaluate_predictions(test_probs, y_test, optimal_tau=optimal_tau)
    
    print("\n" + "="*60)
    print(f"  HASIL EVALUASI TEST SET 60 HARI (EXP21D - {sampling_method.upper()})")
    print("="*60)
    m_opt = eval_results['metrics_optimal_threshold']
    m_def = eval_results['metrics_default_threshold_05']
    m_ind = eval_results['metrics_threshold_independent']
    
    print(f"Threshold Optimal (tau* = {optimal_tau:.3f}):")
    print(f"  • Test F1-Score: {m_opt['f1_score']:.4f}")
    print(f"  • Test Recall  : {m_opt['recall']*100:.2f}% ({m_opt['confusion_matrix']['tp']}/{m_opt['confusion_matrix']['tp']+m_opt['confusion_matrix']['fn']})")
    print(f"  • Test Precision: {m_opt['precision']*100:.2f}% (FP = {m_opt['confusion_matrix']['fp']})")
    print(f"  • Test Accuracy : {m_opt['accuracy']*100:.4f}%")
    print("Threshold Standar (tau = 0.50):")
    print(f"  • Test F1-Score: {m_def['f1_score']:.4f} | Recall: {m_def['recall']*100:.2f}% | Precision: {m_def['precision']*100:.2f}%")
    print("Metrik Independen Threshold:")
    print(f"  • Test AUCPR   : {m_ind['auprc']:.4f}")
    print(f"  • Test ROC-AUC : {m_ind['roc_auc']:.5f}")
    print(f"  • Lift @ 1%    : {m_ind['lift_1_percent']:.2f}x")
    print(f"  • TP @ 500     : {m_ind['tp_at_500']} / 500 ({m_ind['precision_at_500']*100:.1f}%)")
    
    # 12. Audit Keputusan & Feature Importance (Split Gain)
    booster = xgb_model.get_booster()
    score_gain = booster.get_score(importance_type='gain')
    score_weight = booster.get_score(importance_type='weight')
    
    total_gain_tabular = 0.0
    total_gain_gnn = 0.0
    feature_importance_list = []
    
    for i in range(X_train.shape[1]):
        fname = f"f{i}"
        gain = score_gain.get(fname, 0.0)
        weight = score_weight.get(fname, 0)
        
        is_gnn = (i >= feat_dim_tab)
        if is_gnn:
            name = f"gnn_emb_{i - feat_dim_tab}"
            total_gain_gnn += gain
        else:
            name = feature_cols[i]
            total_gain_tabular += gain
            
        feature_importance_list.append({
            "feature_index": i,
            "feature_name": name,
            "is_gnn": is_gnn,
            "gain": float(gain),
            "weight": int(weight)
        })
        
    total_gain_all = total_gain_tabular + total_gain_gnn
    pct_gnn_gain = (total_gain_gnn / total_gain_all * 100) if total_gain_all > 0 else 0.0
    pct_tab_gain = (total_gain_tabular / total_gain_all * 100) if total_gain_all > 0 else 0.0
    
    feature_importance_list.sort(key=lambda x: x['gain'], reverse=True)
    
    print("\nAudit Keputusan Pohon XGBoost:")
    print(f"  • Total Split Gain GNN Embeddings : {total_gain_gnn:,.2f} ({pct_gnn_gain:.2f}%)")
    print(f"  • Total Split Gain Tabular Bersih : {total_gain_tabular:,.2f} ({pct_tab_gain:.2f}%)")
    print("Top 5 Fitur Penyumbang Gain Tertinggi:")
    for rank, f_item in enumerate(feature_importance_list[:5], 1):
        tipe = "GNN" if f_item['is_gnn'] else "Tabular"
        print(f"    {rank}. {f_item['feature_name']} [{tipe}]: Gain = {f_item['gain']:,.2f} (Weight = {f_item['weight']})")
        
    # 13. Studi Ablasi Otomatis (Tabular Only vs GraphSAGE Only vs Hybrid)
    ablation_summary = []
    tab_val_p = tab_test_p = gnn_val_p = gnn_test_p = None
    if cfg.get('ablation', {}).get('run_feature_subsets', True):
        print("\n" + "="*50)
        print("  STUDI ABLASI SUBSET FITUR (GNN LIFT VERIFICATION)")
        print("="*50)
        
        # A. Tabular Only
        print("Mengevaluasi Model: Tabular Only...")
        xgb_tab = XGBClassifier(**xgb_params)
        xgb_tab.fit(X_train[:, :feat_dim_tab], y_train, eval_set=[(X_val[:, :feat_dim_tab], y_val)], verbose=False)
        tab_val_p = xgb_tab.predict_proba(X_val[:, :feat_dim_tab])[:, 1]
        tab_test_p = xgb_tab.predict_proba(X_test[:, :feat_dim_tab])[:, 1]
        xgb_tab.save_model(str(model_dir / f"exp21d_v5_{sampling_method}_tabular_xgb.json"))
        xgb_histories["tabular"] = xgb_tab.evals_result()
        tau_tab, _ = find_optimal_threshold(tab_val_p, y_val, min_precision=min_prec)
        res_tab = evaluate_predictions(tab_test_p, y_test, optimal_tau=tau_tab)
        
        # B. GraphSAGE Only
        print("Mengevaluasi Model: GraphSAGE Only...")
        xgb_gnn = XGBClassifier(**xgb_params)
        xgb_gnn.fit(X_train[:, feat_dim_tab:], y_train, eval_set=[(X_val[:, feat_dim_tab:], y_val)], verbose=False)
        gnn_val_p = xgb_gnn.predict_proba(X_val[:, feat_dim_tab:])[:, 1]
        gnn_test_p = xgb_gnn.predict_proba(X_test[:, feat_dim_tab:])[:, 1]
        xgb_gnn.save_model(str(model_dir / f"exp21d_v5_{sampling_method}_graphsage_embedding_xgb.json"))
        xgb_histories["graphsage_embedding"] = xgb_gnn.evals_result()
        tau_gnn, _ = find_optimal_threshold(gnn_val_p, y_val, min_precision=min_prec)
        res_gnn = evaluate_predictions(gnn_test_p, y_test, optimal_tau=tau_gnn)
        
        ablation_summary = [
            {
                "subset_name": "Tabular Only",
                "optimal_tau": tau_tab,
                "test_f1": res_tab['metrics_optimal_threshold']['f1_score'],
                "test_recall": res_tab['metrics_optimal_threshold']['recall'],
                "test_precision": res_tab['metrics_optimal_threshold']['precision'],
                "test_auprc": res_tab['metrics_threshold_independent']['auprc'],
                "test_roc_auc": res_tab['metrics_threshold_independent']['roc_auc']
            },
            {
                "subset_name": "GraphSAGE Only",
                "optimal_tau": tau_gnn,
                "test_f1": res_gnn['metrics_optimal_threshold']['f1_score'],
                "test_recall": res_gnn['metrics_optimal_threshold']['recall'],
                "test_precision": res_gnn['metrics_optimal_threshold']['precision'],
                "test_auprc": res_gnn['metrics_threshold_independent']['auprc'],
                "test_roc_auc": res_gnn['metrics_threshold_independent']['roc_auc']
            },
            {
                "subset_name": "Hybrid (Golden Master)",
                "optimal_tau": optimal_tau,
                "test_f1": m_opt['f1_score'],
                "test_recall": m_opt['recall'],
                "test_precision": m_opt['precision'],
                "test_auprc": m_ind['auprc'],
                "test_roc_auc": m_ind['roc_auc']
            }
        ]
        
        print("\nRingkasan Studi Ablasi Subset Fitur:")
        for ab in ablation_summary:
            print(f"  • {ab['subset_name']:23s} | F1: {ab['test_f1']:.4f} | Recall: {ab['test_recall']*100:.2f}% | Precision: {ab['test_precision']*100:.2f}% | AUCPR: {ab['test_auprc']:.4f}")
            
    total_pipeline_time = time.time() - start_total_time
    
    # 14. Penyusunan dan Penyimpanan Artefak Hasil
    output_data = {
        "experiment_id": "EXP21D-v5-SamplerCorrectness",
        "experiment_name": "Active Weighted Sampler with Counterpart Degree",
        "version": "v5_sampler_correctness",
        "sampling_method": sampling_method,
        "seed": seed,
        "evaluation_seed": evaluation_seed,
        "feature_mode": feat_cfg.get('mode', 'clean'),
        "feature_cols": feature_cols,
        "num_features_tabular": feat_dim_tab,
        "num_features_gnn": hidden_dim,
        "time_window": tw_cfg,
        "threshold_calibration": tau_metrics,
        "metrics_optimal_threshold": m_opt,
        "metrics_default_threshold_05": m_def,
        "metrics_threshold_independent": m_ind,
        "feature_importance_summary": {
            "total_gain_tabular": float(total_gain_tabular),
            "total_gain_gnn": float(total_gain_gnn),
            "pct_tabular_gain": float(pct_tab_gain),
            "pct_gnn_gain": float(pct_gnn_gain)
        },
        "feature_subsets_ablation": ablation_summary,
        "sampler_correctness": {
            "status": "passed",
            "weight_attr": weight_attr,
            "smoothing_alpha": alpha,
            "counterpart_mapping": {
                "transaction__rev_buys__user": "merchant_degree",
                "transaction__rev_sells__merchant": "user_degree"
            },
            "audit_file": audit_path.name
        },
        "runtime_seconds": {
            "data_preprocessing": data_prep_time,
            "gnn_training": gnn_train_time,
            "xgb_training": xgb_train_time,
            "total_test_inference": inf_test_t,
            "total_pipeline": total_pipeline_time
        },
        "hyperparameters": {
            "hidden_channels": hidden_dim,
            "num_layers": num_layers,
            "aggr": aggr,
            "dropout": dropout,
            "gnn_epochs": epochs,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "best_gnn_epoch": best_epoch,
            "best_gnn_val_auprc": best_val_auprc,
            "num_neighbors": num_neighbors,
            "xgb_params": xgb_params
        }
    }
    
    # Simpan JSON hasil
    results_path = result_dir / f"exp21d_v5_{sampling_method}_results.json"
    write_json(results_path, output_data)
        
    # Simpan JSON feature importance
    fi_path = result_dir / f"exp21d_v5_{sampling_method}_feature_importance.json"
    write_json(fi_path, {
            "total_gain_all": float(total_gain_all),
            "total_gain_tabular": float(total_gain_tabular),
            "total_gain_gnn": float(total_gain_gnn),
            "pct_tabular_gain": float(pct_tab_gain),
            "pct_gnn_gain": float(pct_gnn_gain),
            "top_features": feature_importance_list[:30]
        })
        
    # Simpan JSON riwayat training GNN
    hist_path = result_dir / f"exp21d_v5_{sampling_method}_history.json"
    write_json(hist_path, training_history)
    write_json(result_dir / f"exp21d_v5_{sampling_method}_xgboost_history.json", xgb_histories)

    if cfg.get('output', {}).get('save_feature_matrices', True):
        feature_dir = result_dir / "features" / sampling_method
        feature_dir.mkdir(parents=True, exist_ok=True)
        np.save(feature_dir / "train_hybrid_features.npy", X_train.astype(np.float32, copy=False))
        np.save(feature_dir / "validation_hybrid_features.npy", X_val.astype(np.float32, copy=False))
        np.save(feature_dir / "test_hybrid_features.npy", X_test.astype(np.float32, copy=False))
        np.save(feature_dir / "train_labels.npy", y_train.astype(np.int8, copy=False))
        np.save(feature_dir / "validation_labels.npy", y_val.astype(np.int8, copy=False))
        np.save(feature_dir / "test_labels.npy", y_test.astype(np.int8, copy=False))
        write_json(feature_dir / "feature_contract.json", {
            "tabular_columns": feature_cols,
            "tabular_dimensions": feat_dim_tab,
            "graphsage_embedding_dimensions": hidden_dim,
            "hybrid_dimensions": int(X_train.shape[1]),
            "split_point": feat_dim_tab,
            "dtype": "float32"
        })
        
    # Simpan Prediksi Probabilitas Kontinu NPZ
    npz_path = result_dir / f"exp21d_v5_{sampling_method}_predictions.npz"
    prediction_payload = {
        "val_probs": val_probs,
        "val_labels": y_val.astype(np.int8),
        "test_probs": test_probs,
        "test_labels": y_test.astype(np.int8),
        "optimal_tau": optimal_tau,
        "hybrid_validation_probabilities": val_probs,
        "hybrid_test_probabilities": test_probs,
        "validation_labels": y_val.astype(np.int8),
        "hybrid_optimal_threshold": optimal_tau,
    }
    if tab_val_p is not None:
        prediction_payload.update({
            "tabular_validation_probabilities": tab_val_p,
            "tabular_test_probabilities": tab_test_p,
            "graphsage_validation_probabilities": gnn_val_p,
            "graphsage_test_probabilities": gnn_test_p,
        })
    np.savez_compressed(npz_path, **prediction_payload)

    manifest_path = result_dir / f"exp21d_v5_{sampling_method}_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest.update({
        "status": "complete",
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "best_epoch": best_epoch,
        "best_validation_auprc": best_val_auprc,
        "total_runtime_seconds": total_pipeline_time,
        "test_evaluations": 1,
    })
    write_json(manifest_path, manifest)
    append_event(events_path, "run_completed", method=sampling_method, total_seconds=total_pipeline_time)
    
    print("\n" + "="*80)
    print(f"  EKSEKUSI EXP21D-v5 ({sampling_method.upper()}) BERHASIL SELESAI")
    print(f"  • Berkas Hasil   : {results_path}")
    print(f"  • Berkas Penting : {fi_path}")
    print(f"  • Prediksi Array : {npz_path}")
    print(f"  • Total Waktu    : {total_pipeline_time:.2f} detik (~{total_pipeline_time/60:.1f} menit)")
    print("="*80 + "\n")

if __name__ == "__main__":
    main()



In [ ]:
%%writefile visualize_results.py
import json
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import precision_recall_curve, roc_curve

def setup_plot_style():
    plt.rcParams.update({
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 13,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight'
    })

def plot_pr_roc_curves(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Kurva PR dan ROC Kontinu untuk seluruh metode sampling pada Test Set 60 hari.
    """
    fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(16, 7))
    palette = {
        'uniform': '#1f77b4',
        'topology': '#2ca02c',
        'importance': '#d62728'
    }
    
    has_data = False
    
    for method in methods:
        npz_path = result_dir / f"exp21d_v5_{method}_predictions.npz"
        res_path = result_dir / f"exp21d_v5_{method}_results.json"
        
        if not npz_path.exists() or not res_path.exists():
            continue
            
        data = np.load(npz_path)
        with open(res_path, 'r') as f:
            res = json.load(f)
            
        test_labels = data['test_labels']
        test_probs = data['test_probs']
        
        auprc = res['metrics_threshold_independent']['auprc']
        roc_auc = res['metrics_threshold_independent']['roc_auc']
        f1_opt = res['metrics_optimal_threshold']['f1_score']
        rec_opt = res['metrics_optimal_threshold']['recall']
        prec_opt = res['metrics_optimal_threshold']['precision']
        
        color = palette.get(method, '#333333')
        
        # 1. Kurva PR
        precisions, recalls, _ = precision_recall_curve(test_labels, test_probs)
        ax_pr.plot(recalls, precisions, label=f"{method.capitalize()} (AUPRC = {auprc:.4f}, F1 = {f1_opt:.4f})", color=color, linewidth=2.2)
        ax_pr.scatter([rec_opt], [prec_opt], color=color, s=120, zorder=5, edgecolor='black', linewidth=1.5, marker='*')
        
        # 2. Kurva ROC
        fprs, tprs, _ = roc_curve(test_labels, test_probs)
        ax_roc.plot(fprs, tprs, label=f"{method.capitalize()} (AUROC = {roc_auc:.5f})", color=color, linewidth=2.2)
        has_data = True

    if not has_data:
        plt.close()
        return

    # Baseline & Format PR
    baseline_rate = float(np.mean(test_labels))
    ax_pr.axhline(baseline_rate, color='gray', linestyle='--', label=f'Baseline (Rate = {baseline_rate*100:.3f}%)')
    ax_pr.set_xlabel('Recall (Tingkat Deteksi Kasus Fraud)')
    ax_pr.set_ylabel('Precision (Proporsi Ketepatan Alarm)')
    ax_pr.set_title('Precision-Recall Curves (Test Set 60 Hari)\n* Bintang menunjukkan titik ambang batas optimal tau*', fontweight='bold')
    ax_pr.grid(True, linestyle='--', alpha=0.6)
    ax_pr.legend(loc='lower left')
    ax_pr.set_xlim([0.0, 1.02])
    ax_pr.set_ylim([0.0, 1.05])
    
    # Format ROC
    ax_roc.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Chance (AUC = 0.50)')
    ax_roc.set_xlabel('False Positive Rate (FPR)')
    ax_roc.set_ylabel('True Positive Rate (Recall / TPR)')
    ax_roc.set_title('Receiver Operating Characteristic (ROC) Curves\n(Separasi Probabilitas Kontinu)', fontweight='bold')
    ax_roc.grid(True, linestyle='--', alpha=0.6)
    ax_roc.legend(loc='lower right')
    ax_roc.set_xlim([-0.01, 1.01])
    ax_roc.set_ylim([0.0, 1.02])
    
    plt.tight_layout()
    save_path = output_dir / "exp21d_pr_roc_curves.png"
    plt.savefig(save_path)
    plt.close()
    print(f"Chart tersimpan: {save_path}")

def plot_gnn_lift_comparison(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Studi Ablasi Subset Fitur: Membuktikan kontribusi lift GraphSAGE
    (Tabular Only vs GraphSAGE Only vs Hybrid) untuk metrik F1-Score dan AUCPR.
    """
    lift_data = []
    for method in methods:
        res_path = result_dir / f"exp21d_v5_{method}_results.json"
        if not res_path.exists():
            continue
        with open(res_path, 'r') as f:
            res = json.load(f)
            
        for ab in res.get('feature_subsets_ablation', []):
            lift_data.append({
                "Method": method.capitalize(),
                "Subset": ab['subset_name'],
                "F1": ab['test_f1'],
                "AUCPR": ab['test_auprc'],
                "Recall": ab['test_recall'],
                "Precision": ab['test_precision']
            })
            
    if not lift_data:
        return
        
    df_lift = pd.DataFrame(lift_data)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Palette per subset
    subset_colors = {
        'Tabular Only': '#ff7f0e',
        'GraphSAGE Only': '#2ca02c',
        'Hybrid (Golden Master)': '#1f77b4'
    }
    
    # Plot 1: F1-Score
    pivot_f1 = df_lift.pivot(index='Method', columns='Subset', values='F1')
    pivot_f1 = pivot_f1[['Tabular Only', 'GraphSAGE Only', 'Hybrid (Golden Master)']]
    pivot_f1.plot(kind='bar', ax=ax1, color=[subset_colors[c] for c in pivot_f1.columns], edgecolor='black', width=0.7)
    ax1.set_title('Komparasi F1-Score: Nilai Tambah (Lift) Representasi GraphSAGE', fontweight='bold')
    ax1.set_ylabel('F1-Score pada Test Set')
    ax1.set_ylim([0.80, 1.00])
    ax1.grid(axis='y', linestyle='--', alpha=0.6)
    for container in ax1.containers:
        ax1.bar_label(container, fmt='%.3f', padding=3, fontsize=9)
        
    # Plot 2: AUCPR
    pivot_auc = df_lift.pivot(index='Method', columns='Subset', values='AUCPR')
    pivot_auc = pivot_auc[['Tabular Only', 'GraphSAGE Only', 'Hybrid (Golden Master)']]
    pivot_auc.plot(kind='bar', ax=ax2, color=[subset_colors[c] for c in pivot_auc.columns], edgecolor='black', width=0.7)
    ax2.set_title('Komparasi AUCPR (Presisi Rata-rata Kontinu)', fontweight='bold')
    ax2.set_ylabel('AUCPR pada Test Set')
    ax2.set_ylim([0.80, 1.00])
    ax2.grid(axis='y', linestyle='--', alpha=0.6)
    for container in ax2.containers:
        ax2.bar_label(container, fmt='%.3f', padding=3, fontsize=9)

    plt.tight_layout()
    save_path = output_dir / "exp21d_gnn_lift_comparison.png"
    plt.savefig(save_path)
    plt.close()
    print(f"Chart tersimpan: {save_path}")

def plot_feature_gain_distribution(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Proporsi Split Gain: Membuktikan bahwa pohon XGBoost mengandalkan
    fitur graf GraphSAGE (>85%-95%) dan bebas dari memorization tabular.
    """
    gain_records = []
    for method in methods:
        fi_path = result_dir / f"exp21d_v5_{method}_feature_importance.json"
        if not fi_path.exists():
            continue
        with open(fi_path, 'r') as f:
            data = json.load(f)
        gain_records.append({
            "Method": method.capitalize(),
            "GNN Embeddings": data['pct_gnn_gain'],
            "Clean Tabular": data['pct_tabular_gain']
        })
        
    if not gain_records:
        return
        
    df_g = pd.DataFrame(gain_records).set_index("Method")
    
    fig, ax = plt.subplots(figsize=(10, 5))
    df_g[['GNN Embeddings', 'Clean Tabular']].plot(
        kind='barh', 
        stacked=True, 
        ax=ax, 
        color=['#1f77b4', '#ff7f0e'], 
        edgecolor='black',
        width=0.55
    )
    
    ax.set_title('EXP21D: Proporsi Kontribusi Split Gain XGBoost\n(Restorasi Dominasi Representasi Graf vs Fitur Tabular)', fontweight='bold')
    ax.set_xlabel('Persentase Total Gain (%)')
    ax.set_xlim([0, 100])
    ax.grid(axis='x', linestyle='--', alpha=0.6)
    
    for c in ax.containers:
        ax.bar_label(c, fmt='%.1f%%', label_type='center', color='white', fontweight='bold', fontsize=11)
        
    ax.legend(loc='lower left')
    plt.tight_layout()
    save_path = output_dir / "exp21d_feature_gain_distribution.png"
    plt.savefig(save_path)
    plt.close()
    print(f"Chart tersimpan: {save_path}")

def plot_confusion_matrices(result_dir: Path, output_dir: Path, methods: list):
    """
    Heatmap Matriks Kebingungan (Confusion Matrix) untuk masing-masing metode sampling.
    """
    cms = []
    titles = []
    
    for method in methods:
        res_path = result_dir / f"exp21d_v5_{method}_results.json"
        if not res_path.exists():
            continue
        with open(res_path, 'r') as f:
            res = json.load(f)
        cm = res['metrics_optimal_threshold']['confusion_matrix']
        cms.append(np.array([[cm['tn'], cm['fp']], [cm['fn'], cm['tp']]]))
        titles.append(f"{method.capitalize()} (tau* = {res['metrics_optimal_threshold']['threshold']:.3f})")
        
    if not cms:
        return
        
    n = len(cms)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5))
    if n == 1:
        axes = [axes]
        
    for ax, cm, title in zip(axes, cms, titles):
        ax.imshow(cm, cmap='Blues', interpolation='nearest')
        ax.set_title(title, fontweight='bold', fontsize=12)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Pred Non-Fraud', 'Pred Fraud'])
        ax.set_yticklabels(['True Non-Fraud', 'True Fraud'])
        
        for i in range(2):
            for j in range(2):
                val = cm[i, j]
                text_color = 'white' if val > cm.max() / 2 else 'black'
                ax.text(j, i, f"{val:,}", ha='center', va='center', color=text_color, fontweight='bold', fontsize=12)
                
    plt.tight_layout()
    save_path = output_dir / "exp21d_confusion_matrices.png"
    plt.savefig(save_path)
    plt.close()
    print(f"Chart tersimpan: {save_path}")

def plot_feature_importance_per_method(result_dir: Path, output_dir: Path, methods: list):
    """
    Bar chart top 15 fitur dengan kontribusi split gain terbesar per metode sampling.
    """
    for method in methods:
        fi_path = result_dir / f"exp21d_v5_{method}_feature_importance.json"
        if not fi_path.exists():
            continue
        with open(fi_path, 'r') as f:
            data = json.load(f)
            
        top_f = data.get('top_features', [])[:15]
        if not top_f:
            continue
            
        names = [item['feature_name'] for item in reversed(top_f)]
        gains = [item['gain'] for item in reversed(top_f)]
        colors = ['#1f77b4' if item['is_gnn'] else '#ff7f0e' for item in reversed(top_f)]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(names, gains, color=colors, edgecolor='black', linewidth=0.7)
        ax.set_xlabel('Split Gain Keputusan Pohon')
        ax.set_title(f'EXP21D: Top 15 Fitur XGBoost ({method.upper()})\n'
                     f'GNN Embeddings: {data["pct_gnn_gain"]:.1f}% | Clean Tabular: {data["pct_tabular_gain"]:.1f}%', fontweight='bold')
        ax.grid(axis='x', linestyle='--', alpha=0.6)
        
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#1f77b4', edgecolor='black', label='GraphSAGE Embedding'),
            Patch(facecolor='#ff7f0e', edgecolor='black', label='Clean Tabular Feature')
        ]
        ax.legend(handles=legend_elements, loc='lower right')
        
        plt.tight_layout()
        save_path = output_dir / f"exp21d_{method}_feature_importance.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_learning_curves_per_method(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Kurva Pembelajaran GNN: Loss dan Validation AUPRC per-epoch.
    """
    for method in methods:
        hist_path = result_dir / f"exp21d_v5_{method}_history.json"
        if not hist_path.exists():
            continue
        with open(hist_path, 'r') as f:
            hist = json.load(f)
            
        epochs = list(range(1, len(hist['train_loss']) + 1))
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        ax1.plot(epochs, hist['train_loss'], label='Train Weighted Loss', color='#1f77b4', marker='o')
        ax1.plot(epochs, hist['val_loss'], label='Val Unweighted Loss', color='#ff7f0e', marker='s')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title(f'GNN Loss Dynamics ({method.upper()})', fontweight='bold')
        ax1.grid(True, linestyle='--', alpha=0.6)
        ax1.legend()
        
        ax2.plot(epochs, hist['val_auprc'], label='Validation AUPRC', color='#2ca02c', marker='^', linewidth=2)
        ax2.plot(epochs, hist['val_f1'], label='Validation F1', color='#d62728', linestyle='--', marker='d')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Metric Score')
        ax2.set_title(f'Validation Tracking Metrics ({method.upper()})', fontweight='bold')
        ax2.grid(True, linestyle='--', alpha=0.6)
        ax2.legend()
        
        plt.tight_layout()
        save_path = output_dir / f"exp21d_{method}_learning_curve.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def main():
    parser = argparse.ArgumentParser(description="Generasi Visualisasi Hasil EXP21D")
    parser.add_argument("--result-dir", type=str, default="../../result/exp21d", help="Path direktori hasil")
    parser.add_argument("--output-dir", type=str, default="../../result/exp21d/plots", help="Path direktori penyimpanan grafik")
    args = parser.parse_args()
    
    setup_plot_style()
    result_dir = Path(args.result_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    methods = ["uniform", "topology", "importance"]
    print(f"\n[VISUALISASI] Memulai pembuatan grafik ilmiah dari: {result_dir}")
    print(f"[VISUALISASI] Direktori penyimpanan: {output_dir}\n")
    
    plot_pr_roc_curves(result_dir, output_dir, methods)
    plot_gnn_lift_comparison(result_dir, output_dir, methods)
    plot_feature_gain_distribution(result_dir, output_dir, methods)
    plot_confusion_matrices(result_dir, output_dir, methods)
    plot_feature_importance_per_method(result_dir, output_dir, methods)
    plot_learning_curves_per_method(result_dir, output_dir, methods)
    
    print(f"\n[VISUALISASI] Selesai menghasilkan seluruh chart ilmiah 300 DPI di: {output_dir}\n")

if __name__ == "__main__":
    main()



In [ ]:
%%writefile run_all.py
from __future__ import annotations

import argparse
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml


METHODS = ("uniform", "topology", "importance")


def run(command: list[str], cwd: Path) -> None:
    print(f">>> {' '.join(command)}", flush=True)
    subprocess.run(command, cwd=cwd, check=True)


def aggregate(result_dir: Path) -> None:
    rows, ablations = [], []
    for method in METHODS:
        path = result_dir / f"exp21d_v5_{method}_results.json"
        result = json.loads(path.read_text(encoding="utf-8"))
        optimal = result["metrics_optimal_threshold"]
        independent = result["metrics_threshold_independent"]
        rows.append({
            "sampler": method,
            "seed": result["seed"],
            "optimal_threshold": optimal["threshold"],
            "test_f1": optimal["f1_score"],
            "test_recall": optimal["recall"],
            "test_precision": optimal["precision"],
            "test_auprc": independent["auprc"],
            "test_roc_auc": independent["roc_auc"],
            "false_positive": optimal["confusion_matrix"]["fp"],
            "false_negative": optimal["confusion_matrix"]["fn"],
            "weight_attr": result["sampler_correctness"]["weight_attr"],
            "correctness_status": result["sampler_correctness"]["status"],
        })
        for item in result["feature_subsets_ablation"]:
            ablations.append({"sampler": method, **item})
    frame = pd.DataFrame(rows)
    frame.to_csv(result_dir / "exp21d_v5_master_comparison.csv", index=False)
    pd.DataFrame(ablations).to_csv(result_dir / "exp21d_v5_feature_subsets_comparison.csv", index=False)
    markdown = "# EXP21D-v5 Active Weighted Sampler Correctness\n\n"
    markdown += "Semua run menggunakan counterpart degree dan `weight_attr=sampling_weight`.\n\n"
    markdown += frame.to_markdown(index=False) + "\n"
    (result_dir / "exp21d_v5_summary_report.md").write_text(markdown, encoding="utf-8")


def resolve_kaggle_dataset(config_path: Path) -> None:
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        return
    matches = sorted(kaggle_input.rglob("credit_card_transactions-ibm_v2.csv"))
    if not matches:
        raise FileNotFoundError("credit_card_transactions-ibm_v2.csv tidak ditemukan di /kaggle/input")
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    config["data"]["transactions"] = str(matches[0])
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description="Run EXP21D-v5 single-seed sampler correctness matrix")
    parser.add_argument("--config", default="config.reference.yaml")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default=None)
    parser.add_argument("--nrows", type=int, default=None)
    args = parser.parse_args()
    source = Path(__file__).resolve().parent
    config_path = Path(args.config).resolve()
    resolve_kaggle_dataset(config_path)
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    result_root = Path(config["output"]["result_dir"])
    if not result_root.is_absolute():
        result_root = (config_path.parent / result_root).resolve()
    result_dir = result_root / f"seed_{args.seed}"
    run([sys.executable, "validate_sampler.py"], source)
    for method in METHODS:
        command = [sys.executable, "main.py", "--config", str(config_path), "--sampling", method, "--seed", str(args.seed)]
        if args.device:
            command += ["--device", args.device]
        if args.nrows is not None:
            command += ["--nrows", str(args.nrows)]
        run(command, source)
    aggregate(result_dir)
    run([sys.executable, "visualize_results.py", "--result-dir", str(result_dir), "--output-dir", str(result_dir / "plots")], source)
    print(json.dumps({"status": "complete", "result_dir": str(result_dir)}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
import subprocess
import sys
import torch
torch_version = torch.__version__.split('+')[0]
cuda = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+cu{cuda}.html' if cuda != 'cpu' else f'https://data.pyg.org/whl/torch-{torch_version}+cpu.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric', 'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost', 'tabulate'], check=True)


In [ ]:
!python validate_sampler.py


In [ ]:
!python run_all.py --config config.kaggle.yaml --seed 42
